In [2]:
!git clone https://github.com/ultralytics/ultralytics -b main
%pip install -qe ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 101379, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 101379 (delta 283), reused 219 (delta 219), pack-reused 101066 (from 3)
Receiving objects: 100% (101379/101379), 53.75 MiB | 21.08 MiB/s, done.
Resolving deltas: 100% (76211/76211), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB

**Task - 01**

In [3]:
import yaml

# Define your YAML content as a multi-line string
yaml_content = """
# Dataset paths
path: /kaggle/input/datasets/fateennr/visdrone-dataset/visDrone/VisDrone_Dataset  # dataset root dir
train: /kaggle/input/datasets/fateennr/visdrone-dataset/visDrone/VisDrone_Dataset/VisDrone2019-DET-train/images  # train images
val: /kaggle/input/datasets/fateennr/visdrone-dataset/visDrone/VisDrone_Dataset/VisDrone2019-DET-val/images    # val images
test: /kaggle/input/datasets/fateennr/visdrone-dataset/visDrone/VisDrone_Dataset/VisDrone2019-DET-test-dev/images # test images

#number of classes
nc: 2

# Class names
names: ['person', 'car']
"""

# Write the YAML content to a file
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content)

In [4]:
from pathlib import Path
import pandas as pd
import yaml

# Load dataset.yaml
with open("dataset.yaml", "r") as f:
    data_cfg = yaml.safe_load(f)

train_img_dir = Path(data_cfg["train"])
val_img_dir = Path(data_cfg["val"])
test_img_dir = Path(data_cfg["test"])

# For VisDrone original structure, labels are usually in "annotations"
train_label_dir = train_img_dir.parent / "annotations"
val_label_dir = val_img_dir.parent / "annotations"
test_label_dir = test_img_dir.parent / "annotations"

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp"]

splits = {
    "train": (train_img_dir, train_label_dir),
    "val": (val_img_dir, val_label_dir),
    "test": (test_img_dir, test_label_dir),
}

summary = []

for split_name, (image_dir, label_dir) in splits.items():
    images = []
    for ext in IMAGE_EXTS:
        images.extend(list(image_dir.glob(f"*{ext}")))

    labels = list(label_dir.glob("*.txt"))

    image_stems = set(img.stem for img in images)
    label_stems = set(lbl.stem for lbl in labels)

    images_without_labels = image_stems - label_stems
    labels_without_images = label_stems - image_stems

    summary.append({
        "split": split_name,
        "image_dir_exists": image_dir.exists(),
        "label_dir_exists": label_dir.exists(),
        "images": len(images),
        "labels": len(labels),
        "images_without_labels": len(images_without_labels),
        "labels_without_images": len(labels_without_images),
    })

summary_df = pd.DataFrame(summary)
summary_df

,split,image_dir_exists,label_dir_exists,images,labels,images_without_labels,labels_without_images
0,train,True,False,6471,0,6471,0
1,val,True,False,548,0,548,0
2,test,True,False,1610,0,1610,0


In [3]:
# !yolo train model=yolo26n.pt data=/kaggle/working/dataset.yaml epochs=50 imgsz=960 patience=35 cache=true batch=16 save=true device=0,1 name=run optimizer=SGD

In [ ]:
!yolo train \
  model=yolo26n.pt \
  data=/kaggle/working/dataset.yaml \
  epochs=50 \
  imgsz=960 \
  patience=35 \
  cache=true \
  batch=8 \
  save=true \
  device=0 \
  name=visdrone_run \
  optimizer=SGD \
  mosaic=1.0 \
  close_mosaic=10 \
  scale=0.25 \
  translate=0.08 \
  degrees=3 \
  fliplr=0.5 \
  hsv_s=0.4 \
  hsv_v=0.3

Ultralytics 8.4.51 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=3, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_run-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=35, p

In [ ]:
import os

print("Files in /kaggle/working:")
print(os.listdir("/kaggle/working"))

In [ ]:
import shutil

run_dir = "/kaggle/working/ultralytics/runs/detect/visdrone_run"
zip_path = "/kaggle/working/visdrone_yolo26n_run"

shutil.make_archive(zip_path, "zip", run_dir)

print("Saved:", zip_path + ".zip")

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/visdrone_yolo26n_run.zip")